In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # JobFlow AI — Chunks de Descrições de Vagas
# MAGIC
# MAGIC Este notebook cria chunks das descrições das vagas.
# MAGIC
# MAGIC Os chunks serão usados posteriormente para:
# MAGIC
# MAGIC - busca semântica
# MAGIC - RAG
# MAGIC - ranking de compatibilidade
# MAGIC - explicações do agente

# COMMAND ----------

import re
from datetime import datetime, timezone
from typing import Any

from pyspark.sql import functions as F
from pyspark.sql import types as T

# COMMAND ----------

CATALOG = "workspace"
SCHEMA = "jobflow_ai"

GOLD_JOBS_TABLE = f"{CATALOG}.{SCHEMA}.gold_job_postings"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.gold_job_description_chunks"

CHUNK_MAX_CHARS = 1200
CHUNK_OVERLAP_CHARS = 150
MIN_CHUNK_CHARS = 120

print("=" * 70)
print("JOBFLOW AI — CHUNKS DE DESCRIÇÕES")
print("=" * 70)
print(f"Gold jobs table: {GOLD_JOBS_TABLE}")
print(f"Chunks table: {CHUNKS_TABLE}")
print(f"Chunk max chars: {CHUNK_MAX_CHARS}")
print(f"Chunk overlap chars: {CHUNK_OVERLAP_CHARS}")
print(f"Horário UTC: {datetime.now(timezone.utc).isoformat()}")
print("=" * 70)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Carregar tabela Gold

# COMMAND ----------

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

jobs_df = spark.table(GOLD_JOBS_TABLE)

print(f"Registros na Gold de vagas: {jobs_df.count()}")

display(
    jobs_df.select(
        "job_id",
        "job_title",
        "company_name",
        "job_location",
        "remote_type",
        F.length("job_description").alias("description_length"),
    )
    .orderBy(F.col("description_length").desc())
    .limit(10)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Função de chunking
# MAGIC
# MAGIC A função tenta respeitar sentenças e cria sobreposição pequena entre chunks.

# COMMAND ----------

chunk_schema = T.ArrayType(
    T.StructType(
        [
            T.StructField("chunk_index", T.IntegerType(), False),
            T.StructField("chunk_text", T.StringType(), False),
            T.StructField("chunk_char_length", T.IntegerType(), False),
        ]
    )
)


def normalize_text(text: str | None) -> str:
    if text is None:
        return ""

    cleaned = re.sub(r"\s+", " ", text)
    return cleaned.strip()


def split_into_sentences(text: str) -> list[str]:
    if not text:
        return []

    # Quebra simples por pontuação. Suficiente para MVP.
    pieces = re.split(r"(?<=[.!?])\s+", text)

    return [piece.strip() for piece in pieces if piece and piece.strip()]


def split_long_sentence(sentence: str, max_chars: int) -> list[str]:
    if len(sentence) <= max_chars:
        return [sentence]

    words = sentence.split()
    chunks = []
    current_words = []
    current_length = 0

    for word in words:
        next_length = current_length + len(word) + 1

        if current_words and next_length > max_chars:
            chunks.append(" ".join(current_words))
            current_words = [word]
            current_length = len(word)
        else:
            current_words.append(word)
            current_length = next_length

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


def build_chunks(text: str | None) -> list[dict[str, Any]]:
    normalized = normalize_text(text)

    if not normalized:
        return []

    sentences = split_into_sentences(normalized)

    expanded_sentences = []
    for sentence in sentences:
        expanded_sentences.extend(
            split_long_sentence(sentence, CHUNK_MAX_CHARS)
        )

    chunks = []
    current = ""

    for sentence in expanded_sentences:
        if not current:
            current = sentence
            continue

        candidate = f"{current} {sentence}".strip()

        if len(candidate) <= CHUNK_MAX_CHARS:
            current = candidate
        else:
            if len(current) >= MIN_CHUNK_CHARS:
                chunks.append(current)

            if CHUNK_OVERLAP_CHARS > 0 and current:
                overlap = current[-CHUNK_OVERLAP_CHARS:]
                current = f"{overlap} {sentence}".strip()
            else:
                current = sentence

    if current and len(current) >= MIN_CHUNK_CHARS:
        chunks.append(current)

    return [
        {
            "chunk_index": index,
            "chunk_text": chunk,
            "chunk_char_length": len(chunk),
        }
        for index, chunk in enumerate(chunks)
    ]


build_chunks_udf = F.udf(build_chunks, chunk_schema)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Criar chunks

# COMMAND ----------

jobs_for_chunking_df = (
    jobs_df
    .where(F.col("is_active") == True)
    .where(F.col("job_description").isNotNull())
    .where(F.length(F.col("job_description")) >= MIN_CHUNK_CHARS)
    .select(
        "job_id",
        "source_system",
        "source_job_id",
        "source_job_key",
        "job_title",
        "company_name",
        "job_location",
        "remote_type",
        "tags_text",
        "job_description",
        "job_url",
        "apply_url",
        "published_at",
        "gold_processed_at",
    )
)

chunked_df = (
    jobs_for_chunking_df
    .withColumn("chunks", build_chunks_udf(F.col("job_description")))
    .withColumn("chunk", F.explode(F.col("chunks")))
    .select(
        "job_id",
        "source_system",
        "source_job_id",
        "source_job_key",
        "job_title",
        "company_name",
        "job_location",
        "remote_type",
        "tags_text",
        "job_url",
        "apply_url",
        "published_at",
        F.col("chunk.chunk_index").alias("chunk_index"),
        F.col("chunk.chunk_text").alias("chunk_text"),
        F.col("chunk.chunk_char_length").alias("chunk_char_length"),
        "gold_processed_at",
    )
    .withColumn(
        "chunk_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("job_id"),
                F.col("chunk_index").cast("string"),
                F.col("chunk_text"),
            ),
            256,
        ),
    )
    .withColumn(
        "chunk_token_estimate",
        F.ceil(F.col("chunk_char_length") / F.lit(4)),
    )
    .withColumn(
        "chunk_metadata_text",
        F.concat_ws(
            " | ",
            F.concat(F.lit("Title: "), F.col("job_title")),
            F.concat(F.lit("Company: "), F.col("company_name")),
            F.concat(F.lit("Location: "), F.col("job_location")),
            F.concat(F.lit("Remote type: "), F.col("remote_type")),
            F.concat(F.lit("Tags: "), F.col("tags_text")),
        ),
    )
    .withColumn(
        "embedding_text",
        F.concat_ws(
            "\n\n",
            F.col("chunk_metadata_text"),
            F.col("chunk_text"),
        ),
    )
    .withColumn(
        "chunk_created_at",
        F.current_timestamp(),
    )
    .select(
        "chunk_id",
        "job_id",
        "source_system",
        "source_job_id",
        "source_job_key",
        "job_title",
        "company_name",
        "job_location",
        "remote_type",
        "tags_text",
        "job_url",
        "apply_url",
        "published_at",
        "chunk_index",
        "chunk_text",
        "chunk_char_length",
        "chunk_token_estimate",
        "chunk_metadata_text",
        "embedding_text",
        "gold_processed_at",
        "chunk_created_at",
    )
)

display(chunked_df.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Gravar tabela de chunks

# COMMAND ----------

(
    chunked_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CHUNKS_TABLE)
)

print(f"OK: tabela de chunks gravada em {CHUNKS_TABLE}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Validações

# COMMAND ----------

chunks_df = spark.table(CHUNKS_TABLE)

summary_df = chunks_df.agg(
    F.count("*").alias("total_chunks"),
    F.countDistinct("chunk_id").alias("distinct_chunk_ids"),
    F.countDistinct("job_id").alias("jobs_with_chunks"),
    F.round(F.avg("chunk_char_length"), 2).alias("avg_chunk_chars"),
    F.min("chunk_char_length").alias("min_chunk_chars"),
    F.max("chunk_char_length").alias("max_chunk_chars"),
    F.round(F.avg("chunk_token_estimate"), 2).alias("avg_token_estimate"),
)

display(summary_df)

quality_df = chunks_df.select(
    F.sum(F.when(F.col("chunk_id").isNull(), 1).otherwise(0)).alias("missing_chunk_id"),
    F.sum(F.when(F.col("job_id").isNull(), 1).otherwise(0)).alias("missing_job_id"),
    F.sum(F.when(F.col("chunk_text").isNull() | (F.col("chunk_text") == ""), 1).otherwise(0)).alias("missing_chunk_text"),
    F.sum(F.when(F.col("embedding_text").isNull() | (F.col("embedding_text") == ""), 1).otherwise(0)).alias("missing_embedding_text"),
)

display(quality_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Distribuição de chunks por vaga

# COMMAND ----------

display(
    chunks_df.groupBy(
        "job_id",
        "job_title",
        "company_name",
    )
    .agg(
        F.count("*").alias("chunks"),
        F.sum("chunk_char_length").alias("total_chunk_chars"),
    )
    .orderBy(F.col("chunks").desc(), F.col("total_chunk_chars").desc())
    .limit(20)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Exemplos de chunks

# COMMAND ----------

display(
    chunks_df.select(
        "chunk_id",
        "job_title",
        "company_name",
        "chunk_index",
        "chunk_char_length",
        F.substring("embedding_text", 1, 1200).alias("embedding_text_preview"),
    )
    .orderBy("job_title", "chunk_index")
    .limit(10)
)

# COMMAND ----------

print()
print("=" * 70)
print("RESULTADO: CHUNKS DE DESCRIÇÕES CONCLUÍDOS")
print("=" * 70)
print(f"chunks table: {CHUNKS_TABLE}")
print(f"total chunks: {chunks_df.count()}")
print(f"jobs with chunks: {chunks_df.select('job_id').distinct().count()}")
print("=" * 70)